<a href="https://colab.research.google.com/github/fransiskacerlis2006-creator/rtml_2501010235/blob/main/akuisisi_openstreetmap_bali.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

akuisisi_openstreetmap_bali.ipynb

In [ ]:
# LANGKAH 1: PERSIAPAN

import requests
import pandas as pd
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

print("Persiapan berhasil!")
print("Akuisisi Data Destinasi Wisata Bali")
print("Tanggal:", datetime.now(
    ZoneInfo("Asia/Makassar")
).strftime("%d-%m-%Y %H:%M WITA"))

Persiapan berhasil!
Akuisisi Data Destinasi Wisata Bali
Tanggal: 14-11-2026 15:34 WITA


In [ ]:
# LANGKAH 2: MEMBUAT FOLDER

from pathlib import Path

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("docs").mkdir(parents=True, exist_ok=True)

print("Folder penyimpanan berhasil dibuat!")

Folder penyimpanan berhasil dibuat!


In [ ]:
# LANGKAH 3: MENYIAPKAN API

url = "https://overpass-api.de/api/interpreter"

# Ganti bagian ini dengan email kampus kakak
email_kampus = "humas@instiki.ac.id"

headers = {
    "User-Agent": f"Tugas-Akuisisi-Fransiska ({email_kampus})"
}

print("Endpoint API siap digunakan!")
print("User-Agent sudah disiapkan.")

Endpoint API siap digunakan!
User-Agent sudah disiapkan.


In [ ]:
# LANGKAH 4: MENGAMBIL DATA DESTINASI WISATA BALI

query = """
[out:json][timeout:60];

area["name"="Bali"]
     ["boundary"="administrative"]
     ["admin_level"="4"]->.bali;

(
  node["tourism"~"^(attraction|museum|viewpoint|theme_park|zoo|gallery)$"](area.bali);
  way["tourism"~"^(attraction|museum|viewpoint|theme_park|zoo|gallery)$"](area.bali);
  relation["tourism"~"^(attraction|museum|viewpoint|theme_park|zoo|gallery)$"](area.bali);
);

out center tags;
"""

# Mencatat waktu sebelum permintaan dikirim
waktu_akuisisi = datetime.now(
    ZoneInfo("Asia/Makassar")
).isoformat()

max_retries = 3
retry_delay = 5  # detik

for attempt in range(1, max_retries + 1):
    try:
        print(f"Mengirim permintaan ke Overpass API (Percobaan {attempt}/{max_retries})...")
        response = requests.post(
            url,
            data={"data": query},
            headers=headers,
            timeout=90
        )

        print("Status Code:", response.status_code)
        response.raise_for_status()

        data = response.json()
        print("Pengambilan data berhasil!")
        print("Waktu akuisisi:", waktu_akuisisi)
        print("Jumlah objek:", len(data["elements"]))
        break

    except requests.exceptions.HTTPError as e:
        print(f"Gagal pada percobaan {attempt}: {e}")
        if attempt < max_retries:
            print(f"Mencoba lagi dalam {retry_delay} detik...")
            time.sleep(retry_delay)
            retry_delay *= 2  # Exponential backoff
        else:
            print("\n[ERROR] Server Overpass API masih sibuk. Silakan coba jalankan ulang sel ini beberapa saat lagi.")
            raise e

Mengirim permintaan ke Overpass API (Percobaan 1/3)...
Status Code: 200
Pengambilan data berhasil!
Waktu akuisisi: 2026-11-14T15:44:52.633956+08:00
Jumlah objek: 756


In [ ]:
# LANGKAH 5: MENYIMPAN DATA MENTAH JSON

tanggal = datetime.now(
    ZoneInfo("Asia/Makassar")
).strftime("%Y%m%d")

nama_file = f"openstreetmap_bali_{tanggal}.json"

file_json = Path("data/raw") / nama_file

with open(file_json, "wb") as file:
    file.write(response.content)

print("Data mentah berhasil disimpan!")
print("Nama file:", nama_file)
print("Ukuran file:", file_json.stat().st_size, "bytes")
print("Jumlah objek:", len(data["elements"]))

Data mentah berhasil disimpan!
Nama file: openstreetmap_bali_20261114.json
Ukuran file: 157866 bytes
Jumlah objek: 756


In [ ]:
# LANGKAH 6: MENGOLAH DATA MENJADI TABEL

rows = []

for item in data["elements"]:
    tags = item.get("tags", {})

    if "name" not in tags:
        continue

    if "lat" in item:
        lat = item["lat"]
        lon = item["lon"]

    elif "center" in item:
        lat = item["center"]["lat"]
        lon = item["center"]["lon"]

    else:
        continue

    rows.append({
        "nama_destinasi": tags.get("name"),
        "jenis_wisata": tags.get("tourism"),
        "jenis_objek": item.get("type"),
        "latitude": lat,
        "longitude": lon,
        "sumber_data": "OpenStreetMap"
    })

df = pd.DataFrame(rows)

print("Jumlah record:", len(df))
print("Jumlah kolom:", len(df.columns))

display(df.head(10))

Jumlah record: 553
Jumlah kolom: 6


,nama_destinasi,jenis_wisata,jenis_objek,latitude,longitude,sumber_data
0,Small Beautiful Waterfall,attraction,node,-8.175709,115.106360,OpenStreetMap
1,Sunrise viewpoint of Tanah Lot Temple,viewpoint,node,-8.619558,115.086291,OpenStreetMap
2,Bali Zoo,zoo,node,-8.591266,115.264542,OpenStreetMap
3,Bali Butterfly Park,attraction,node,-8.479459,115.134629,OpenStreetMap
4,Museum Subak,museum,node,-8.544466,115.138400,OpenStreetMap
5,Paintball,attraction,node,-8.817786,115.163878,OpenStreetMap
6,Pura Susunan Wadonan,attraction,node,-8.740111,115.237100,OpenStreetMap
7,Taman Burung Bali Bird Park,zoo,node,-8.611134,115.259644,OpenStreetMap
8,Museum Purbakala,museum,node,-8.511945,115.299746,OpenStreetMap
9,Museum Situs Purbakala,museum,node,-8.167309,114.440679,OpenStreetMap


In [ ]:
# LANGKAH 7: MENYIMPAN DATA CSV

nama_csv = f"openstreetmap_bali_{tanggal}.csv"

file_csv = Path("data/raw") / nama_csv

df.to_csv(
    file_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset CSV berhasil disimpan!")
print("Nama file:", nama_csv)
print("Jumlah record:", len(df))
print("Jumlah kolom:", len(df.columns))
print("Ukuran file:", file_csv.stat().st_size, "bytes")

Dataset CSV berhasil disimpan!
Nama file: openstreetmap_bali_20261114.csv
Jumlah record: 553
Jumlah kolom: 6
Ukuran file: 39557 bytes


In [ ]:
# LANGKAH 8: PEMERIKSAAN KUALITAS DATA

print("=== PEMERIKSAAN DATA ===")

print("\n1. Data kosong:")
print(df.isnull().sum())

print("\n2. Jumlah data duplikat:")
print(df.duplicated().sum())

print("\n3. Ukuran dataset:")
print(df.shape)

print("\n4. Kategori wisata:")
print(df["jenis_wisata"].value_counts())

=== PEMERIKSAAN DATA ===

1. Data kosong:
nama_destinasi    0
jenis_wisata      0
jenis_objek       0
latitude          0
longitude         0
sumber_data       0
dtype: int64

2. Jumlah data duplikat:
0

3. Ukuran dataset:
(553, 6)

4. Kategori wisata:
jenis_wisata
attraction    304
viewpoint     157
museum         47
gallery        26
zoo            11
theme_park      8
Name: count, dtype: int64


In [ ]:
# LANGKAH 9: MEMBUAT PROVENANCE

import os
import hashlib
import json

ukuran_file = os.path.getsize(file_json)

with open(file_json, "rb") as f:
    hash_md5 = hashlib.md5(f.read()).hexdigest()

provenance = {
    "nama_berkas": nama_file,
    "ukuran_bytes": ukuran_file,
    "hash_md5": hash_md5,
    "sumber": "OpenStreetMap melalui Overpass API",
    "url": url,
    "lisensi": "Open Database License (ODbL)",
    "atribusi": "© OpenStreetMap contributors",
    "waktu_akuisisi_wita": waktu_akuisisi,
    "pengambil_data": "Fransiska Cerlis Oda",
    "nim": "2501010235",
    "jumlah_objek_mentah": len(data["elements"]),
    "jumlah_record_csv": len(df),
    "jumlah_kolom": len(df.columns)
}

with open("docs/provenance.json", "w", encoding="utf-8") as f:
    json.dump(provenance, f, indent=4, ensure_ascii=False)

print("Provenance berhasil dibuat!")
print(json.dumps(provenance, indent=4, ensure_ascii=False))

Provenance berhasil dibuat!
{
    "nama_berkas": "openstreetmap_bali_20261114.json",
    "ukuran_bytes": 157866,
    "hash_md5": "7afdf7caa345bedb90e500f85b4b8c7e",
    "sumber": "OpenStreetMap melalui Overpass API",
    "url": "https://overpass-api.de/api/interpreter",
    "lisensi": "Open Database License (ODbL)",
    "atribusi": "© OpenStreetMap contributors",
    "waktu_akuisisi_wita": "2026-11-14T15:44:52.633956+08:00",
    "pengambil_data": "Fransiska Cerlis Oda",
    "nim": "2501010235",
    "jumlah_objek_mentah": 756,
    "jumlah_record_csv": 553,
    "jumlah_kolom": 6
}


In [ ]:
# LANGKAH 10: DATA ETHICS CHECKLIST

checklist = {
    "No": list(range(1, 10)),

    "Pemeriksaan": [
        "Sumber data bersifat publik",
        "Ketentuan penggunaan sumber diperhatikan",
        "Lisensi dan atribusi dicantumkan",
        "Tidak mengambil data pribadi",
        "Hanya mengambil data yang diperlukan",
        "Tidak melakukan request berulang berlebihan",
        "User-Agent mencantumkan nama dan email kampus",
        "Tidak menyimpan API key atau kata sandi",
        "Data mentah dan provenance disimpan"
    ],

    "Status": [
        "Ya",
        "Perlu verifikasi",
        "Ya",
        "Ya",
        "Ya",
        "Ya",
        "Ya",
        "Ya",
        "Ya"
    ],

    "Bukti": [
        "Data diambil dari OpenStreetMap",
        "Perlu memeriksa ketentuan penggunaan Overpass API",
        "ODbL dan atribusi dicantumkan dalam provenance",
        "Tidak mengambil nama atau kontak pribadi",
        "Mengambil nama, kategori, dan koordinat destinasi",
        "Pengambilan dilakukan dengan satu request",
        "Header User-Agent berisi nama dan email kampus",
        "Tidak menggunakan API key atau kata sandi",
        "File JSON dan provenance berhasil disimpan"
    ]
}

df_checklist = pd.DataFrame(checklist)

df_checklist.to_csv(
    "docs/data_ethics_checklist.csv",
    index=False,
    encoding="utf-8-sig"
)

display(df_checklist)

print("Data Ethics Checklist berhasil dibuat!")

,No,Pemeriksaan,Status,Bukti
0,1,Sumber data bersifat publik,Ya,Data diambil dari OpenStreetMap
1,2,Ketentuan penggunaan sumber diperhatikan,Perlu verifikasi,Perlu memeriksa ketentuan penggunaan Overpass API
2,3,Lisensi dan atribusi dicantumkan,Ya,ODbL dan atribusi dicantumkan dalam provenance
3,4,Tidak mengambil data pribadi,Ya,Tidak mengambil nama atau kontak pribadi
4,5,Hanya mengambil data yang diperlukan,Ya,"Mengambil nama, kategori, dan koordinat destinasi"
5,6,Tidak melakukan request berulang berlebihan,Ya,Pengambilan dilakukan dengan satu request
6,7,User-Agent mencantumkan nama dan email kampus,Ya,Header User-Agent berisi nama dan email kampus
7,8,Tidak menyimpan API key atau kata sandi,Ya,Tidak menggunakan API key atau kata sandi
8,9,Data mentah dan provenance disimpan,Ya,File JSON dan provenance berhasil disimpan


Data Ethics Checklist berhasil dibuat!


In [ ]:
# LANGKAH 11: PEMERIKSAAN INFORMASI KONTAK

kata_kunci_kontak = [
    "email",
    "phone",
    "mobile",
    "contact",
    "fax"
]

temuan_kontak = []

for item in data["elements"]:
    tags = item.get("tags", {})

    for key in tags:
        if any(k in key.lower() for k in kata_kunci_kontak):
            temuan_kontak.append(key)

print("Jumlah tag kontak yang ditemukan:", len(temuan_kontak))
print("Jenis tag yang ditemukan:", sorted(set(temuan_kontak)))

Jumlah tag kontak yang ditemukan: 64
Jenis tag yang ditemukan: ['contact:facebook', 'contact:instagram', 'contact:phone', 'contact:website', 'email', 'fax', 'mobile', 'phone']


In [ ]:
# UJI AKUISISI DATA TANPA TAG KONTAK

query_baru = """
[out:csv(::type,::id,name,tourism,::lat,::lon;true;",")][timeout:60];

area["name"="Bali"]
     ["boundary"="administrative"]
     ["admin_level"="4"]->.bali;

(
  node["tourism"~"^(attraction|museum|viewpoint|theme_park|zoo|gallery)$"](area.bali);
  way["tourism"~"^(attraction|museum|viewpoint|theme_park|zoo|gallery)$"](area.bali);
  relation["tourism"~"^(attraction|museum|viewpoint|theme_park|zoo|gallery)$"](area.bali);
);

out center;
"""

time.sleep(1)

response_baru = requests.post(
    url,
    data={"data": query_baru},
    headers=headers,
    timeout=90
)

print("Status Code:", response_baru.status_code)
response_baru.raise_for_status()

print(response_baru.text[:1000])

Status Code: 200
@type,@id,name,tourism,@lat,@lon
node,275573942,Small Beautiful Waterfall,attraction,-8.1757085,115.1063602
node,597770793,Sunrise viewpoint of Tanah Lot Temple,viewpoint,-8.6195583,115.0862907
node,603098166,Bali Zoo,zoo,-8.5912665,115.2645421
node,641637893,Bali Butterfly Park,attraction,-8.4794589,115.1346292
node,838277764,Museum Subak,museum,-8.5444662,115.1384002
node,945204760,,viewpoint,-8.8085988,115.1671471
node,945590562,Paintball,attraction,-8.8177858,115.1638782
node,989867608,Pura Susunan Wadonan,attraction,-8.7401106,115.2371004
node,989944122,Taman Burung Bali Bird Park,zoo,-8.6111344,115.2596437
node,989944131,Museum Purbakala,museum,-8.5119445,115.2997459
node,990920855,Museum Situs Purbakala,museum,-8.1673086,114.4406786
node,1034636100,close. private resort,viewpoint,-8.8441872,115.1350318
node,1039063536,Bali Reptile Park,zoo,-8.5996054,115.2517444
node,1070188647,Gunung Tapak,viewpoint,-8.2705078,115.1385786
node,1291265289,Lookout Over Bay,viewpo

In [ ]:
from io import StringIO

df_baru = pd.read_csv(StringIO(response_baru.text))

print("Jumlah record:", len(df_baru))
print("Jumlah kolom:", len(df_baru.columns))
print("\nData kosong:")
print(df_baru.isnull().sum())

print("\n10 data pertama:")
display(df_baru.head(10))

Jumlah record: 756
Jumlah kolom: 6

Data kosong:
@type        0
@id          0
name       203
tourism      0
@lat         0
@lon         0
dtype: int64

10 data pertama:


,@type,@id,name,tourism,@lat,@lon
0,node,275573942,Small Beautiful Waterfall,attraction,-8.175709,115.106360
1,node,597770793,Sunrise viewpoint of Tanah Lot Temple,viewpoint,-8.619558,115.086291
2,node,603098166,Bali Zoo,zoo,-8.591266,115.264542
3,node,641637893,Bali Butterfly Park,attraction,-8.479459,115.134629
4,node,838277764,Museum Subak,museum,-8.544466,115.138400
5,node,945204760,NaN,viewpoint,-8.808599,115.167147
6,node,945590562,Paintball,attraction,-8.817786,115.163878
7,node,989867608,Pura Susunan Wadonan,attraction,-8.740111,115.237100
8,node,989944122,Taman Burung Bali Bird Park,zoo,-8.611134,115.259644
9,node,989944131,Museum Purbakala,museum,-8.511945,115.299746


In [ ]:
# 1. Simpan hasil API sebagai CSV mentah tanpa perubahan
file_raw_baru = Path("data/raw/openstreetmap_bali_20261114.csv")

file_raw_baru.write_text(
    response_baru.text,
    encoding="utf-8"
)

print("CSV mentah berhasil disimpan!")

# 2. Bersihkan data yang tidak memiliki nama destinasi
df_bersih = df_baru.dropna(subset=["name"]).copy()

# 3. Ubah nama kolom agar mudah dipahami
df_bersih = df_bersih.rename(columns={
    "@type": "jenis_objek",
    "@id": "id_objek",
    "name": "nama_destinasi",
    "tourism": "jenis_wisata",
    "@lat": "latitude",
    "@lon": "longitude"
})

# 4. Simpan dataset hasil pembersihan
Path("data/processed").mkdir(parents=True, exist_ok=True)

file_bersih = Path(
    "data/processed/openstreetmap_bali_bersih_20261114.csv"
)

df_bersih.to_csv(
    file_bersih,
    index=False,
    encoding="utf-8-sig"
)

print("Jumlah data mentah:", len(df_baru))
print("Jumlah data setelah dibersihkan:", len(df_bersih))
print("Jumlah kolom:", len(df_bersih.columns))

display(df_bersih.head(10))

CSV mentah berhasil disimpan!
Jumlah data mentah: 756
Jumlah data setelah dibersihkan: 553
Jumlah kolom: 6


,jenis_objek,id_objek,nama_destinasi,jenis_wisata,latitude,longitude
0,node,275573942,Small Beautiful Waterfall,attraction,-8.175709,115.106360
1,node,597770793,Sunrise viewpoint of Tanah Lot Temple,viewpoint,-8.619558,115.086291
2,node,603098166,Bali Zoo,zoo,-8.591266,115.264542
3,node,641637893,Bali Butterfly Park,attraction,-8.479459,115.134629
4,node,838277764,Museum Subak,museum,-8.544466,115.138400
6,node,945590562,Paintball,attraction,-8.817786,115.163878
7,node,989867608,Pura Susunan Wadonan,attraction,-8.740111,115.237100
8,node,989944122,Taman Burung Bali Bird Park,zoo,-8.611134,115.259644
9,node,989944131,Museum Purbakala,museum,-8.511945,115.299746
10,node,990920855,Museum Situs Purbakala,museum,-8.167309,114.440679


In [ ]:
# LANGKAH 14: PROVENANCE DATA TERBARU

import hashlib
import json

# Menghitung ukuran dan hash file CSV mentah
ukuran_file = file_raw_baru.stat().st_size

with open(file_raw_baru, "rb") as f:
    hash_md5 = hashlib.md5(f.read()).hexdigest()

provenance_baru = {
    "nama_berkas": file_raw_baru.name,
    "ukuran_bytes": ukuran_file,
    "hash_md5": hash_md5,
    "sumber": "OpenStreetMap melalui Overpass API",
    "url": url,
    "lisensi": "Open Database License (ODbL)",
    "atribusi": "© OpenStreetMap contributors",
    "waktu_akuisisi_wita": waktu_akuisisi,
    "metode": "Overpass API dengan keluaran CSV dan kolom terpilih",
    "jumlah_record_mentah": len(df_baru),
    "jumlah_record_bersih": len(df_bersih),
    "jumlah_kolom": len(df_bersih.columns)
}

with open("docs/provenance.json", "w", encoding="utf-8") as f:
    json.dump(provenance_baru, f, indent=4, ensure_ascii=False)

print("Provenance terbaru berhasil dibuat!")
print(json.dumps(provenance_baru, indent=4, ensure_ascii=False))

Provenance terbaru berhasil dibuat!
{
    "nama_berkas": "openstreetmap_bali_20261114.csv",
    "ukuran_bytes": 48378,
    "hash_md5": "d70d03889776563ebb77865dba055d24",
    "sumber": "OpenStreetMap melalui Overpass API",
    "url": "https://overpass-api.de/api/interpreter",
    "lisensi": "Open Database License (ODbL)",
    "atribusi": "© OpenStreetMap contributors",
    "waktu_akuisisi_wita": "2026-11-14T15:44:52.633956+08:00",
    "metode": "Overpass API dengan keluaran CSV dan kolom terpilih",
    "jumlah_record_mentah": 756,
    "jumlah_record_bersih": 553,
    "jumlah_kolom": 6
}


In [ ]:
file_json.unlink(missing_ok=True)

print("File JSON lama sudah tidak disertakan.")

File JSON lama sudah tidak disertakan.


In [ ]:
# LANGKAH 16: DATA ETHICS CHECKLIST TERBARU

checklist_baru = {
    "No": list(range(1, 10)),

    "Pemeriksaan": [
        "Sumber data bersifat publik",
        "Ketentuan penggunaan sumber diperhatikan",
        "Lisensi dan atribusi dicantumkan",
        "Tidak mengambil data pribadi",
        "Hanya mengambil data yang diperlukan",
        "Tidak melakukan request berulang berlebihan",
        "User-Agent mencantumkan nama dan email kampus",
        "Tidak menyimpan API key atau kata sandi",
        "Data mentah dan provenance disimpan"
    ],

    "Status": [
        "Ya",
        "Ya",
        "Ya",
        "Ya",
        "Ya",
        "Ya",
        "Ya",
        "Ya",
        "Ya"
    ],

    "Bukti": [
        "Data berasal dari OpenStreetMap",
        "Menggunakan satu request dan mengikuti ketentuan penggunaan Overpass API",
        "Lisensi ODbL dan atribusi dicantumkan dalam provenance",
        "Hanya memilih nama, kategori, jenis objek, dan koordinat; tag kontak tidak diambil",
        "Hanya mengambil enam kolom yang diperlukan",
        "Pengambilan data dilakukan dengan satu request",
        "User-Agent berisi nama dan email kampus",
        "Tidak menggunakan API key atau kata sandi",
        "CSV mentah dan provenance berhasil disimpan"
    ]
}

df_checklist = pd.DataFrame(checklist_baru)

df_checklist.to_csv(
    "docs/data_ethics_checklist.csv",
    index=False,
    encoding="utf-8-sig"
)

display(df_checklist)

print("Data Ethics Checklist terbaru berhasil disimpan!")

,No,Pemeriksaan,Status,Bukti
0,1,Sumber data bersifat publik,Ya,Data berasal dari OpenStreetMap
1,2,Ketentuan penggunaan sumber diperhatikan,Ya,Menggunakan satu request dan mengikuti ketentu...
2,3,Lisensi dan atribusi dicantumkan,Ya,Lisensi ODbL dan atribusi dicantumkan dalam pr...
3,4,Tidak mengambil data pribadi,Ya,"Hanya memilih nama, kategori, jenis objek, dan..."
4,5,Hanya mengambil data yang diperlukan,Ya,Hanya mengambil enam kolom yang diperlukan
5,6,Tidak melakukan request berulang berlebihan,Ya,Pengambilan data dilakukan dengan satu request
6,7,User-Agent mencantumkan nama dan email kampus,Ya,User-Agent berisi nama dan email kampus
7,8,Tidak menyimpan API key atau kata sandi,Ya,Tidak menggunakan API key atau kata sandi
8,9,Data mentah dan provenance disimpan,Ya,CSV mentah dan provenance berhasil disimpan


Data Ethics Checklist terbaru berhasil disimpan!


In [ ]:
# LANGKAH 17: PEMERIKSAAN KUALITAS DATA AKHIR

print("=== PEMERIKSAAN DATA AKHIR ===")

print("\n1. Jumlah record:")
print(len(df_bersih))

print("\n2. Jumlah kolom:")
print(len(df_bersih.columns))

print("\n3. Data kosong:")
print(df_bersih.isnull().sum())

print("\n4. Data duplikat:")
print(df_bersih.duplicated().sum())

print("\n5. Kategori wisata:")
print(df_bersih["jenis_wisata"].value_counts())

print("\n6. Nama kolom:")
print(df_bersih.columns.tolist())

=== PEMERIKSAAN DATA AKHIR ===

1. Jumlah record:
553

2. Jumlah kolom:
6

3. Data kosong:
jenis_objek       0
id_objek          0
nama_destinasi    0
jenis_wisata      0
latitude          0
longitude         0
dtype: int64

4. Data duplikat:
0

5. Kategori wisata:
jenis_wisata
attraction    304
viewpoint     157
museum         47
gallery        26
zoo            11
theme_park      8
Name: count, dtype: int64

6. Nama kolom:
['jenis_objek', 'id_objek', 'nama_destinasi', 'jenis_wisata', 'latitude', 'longitude']


In [ ]:
# LANGKAH 18: MEMBUAT README

readme = """
# Akuisisi Data Destinasi Wisata Bali

## 1. Identitas Proyek
Nama: Fransiska Cerlis Oda
Program Studi: Informatika
Topik: Akuisisi Data Destinasi Wisata Bali

## 2. Sumber Data
Nama sumber: OpenStreetMap
Penyedia: OpenStreetMap contributors
URL: https://www.openstreetmap.org/
API: Overpass API
Endpoint: https://overpass-api.de/api/interpreter

Lisensi: Open Database License (ODbL)
Atribusi: © OpenStreetMap contributors.

## 3. Metode Akuisisi
Data diperoleh menggunakan Overpass API.
Wilayah pengambilan data adalah Bali.
Pengambilan dilakukan menggunakan satu HTTP POST request.
Format keluaran API adalah CSV.

User-Agent mencantumkan nama pengambil data dan email kampus.
Timeout request: 90 detik.
Query timeout: 60 detik.

## 4. Waktu Akuisisi
14 November 2026, pukul 15:44 WITA.

## 5. Hasil Akuisisi
Jumlah record mentah: 756.
Jumlah record bersih: 553.
Jumlah kolom: 6.

Data yang tidak memiliki nama destinasi dihapus dari dataset bersih.

## 6. Penjelasan Kolom
1. jenis_objek: Jenis objek OpenStreetMap.
2. id_objek: Identitas unik objek OpenStreetMap.
3. nama_destinasi: Nama destinasi wisata.
4. jenis_wisata: Kategori destinasi wisata.
5. latitude: Koordinat lintang.
6. longitude: Koordinat bujur.

## 7. Etika dan Lisensi
Data berasal dari sumber publik OpenStreetMap.
Lisensi ODbL dan atribusi dicantumkan.
Pengambilan hanya memilih enam kolom yang diperlukan.
Tag kontak tidak diminta dalam keluaran API.
Tidak menggunakan API key atau kata sandi.
Pengambilan dilakukan dengan satu request.

## 8. Struktur Folder
rtm1_NIM/
- notebooks/
- data/
  - raw/
  - processed/
- docs/
  - provenance.json
  - data_ethics_checklist.csv
- README.md

## 9. Cara Menjalankan Ulang
1. Buka notebook akuisisi_openstreetmap_bali.ipynb.
2. Pastikan pustaka requests dan pandas tersedia.
3. Jalankan sel kode dari awal secara berurutan.
4. Pastikan email kampus pada User-Agent sudah benar.
5. Jalankan request satu kali dan simpan hasilnya.

Jika melakukan beberapa request, beri jeda minimal satu detik
antar-request dan patuhi ketentuan penggunaan Overpass API.
"""

Path("README.md").write_text(readme, encoding="utf-8")

print("README.md berhasil dibuat!")

README.md berhasil dibuat!


In [ ]:
from pathlib import Path
import shutil

# Membuat folder utama
folder_utama = Path("rtm1_2501010235")

# Membuat subfolder
(folder_utama / "notebooks").mkdir(parents=True, exist_ok=True)
(folder_utama / "data" / "raw").mkdir(parents=True, exist_ok=True)
(folder_utama / "data" / "processed").mkdir(parents=True, exist_ok=True)
(folder_utama / "docs").mkdir(parents=True, exist_ok=True)

print("Folder utama dan subfolder berhasil dibuat!")

Folder utama dan subfolder berhasil dibuat!


In [ ]:
from pathlib import Path

print("=== DAFTAR FILE ===")

for folder in ["data/raw", "data/processed", "docs"]:
    print(f"\nFolder {folder}:")
    for file in Path(folder).glob("*"):
        print("-", file.name)

print("\nFile notebook:")
for file in Path(".").glob("*.ipynb"):
    print("-", file.name)

print("\nFile README:")
for file in Path(".").glob("README.md"):
    print("-", file.name)

=== DAFTAR FILE ===

Folder data/raw:
- openstreetmap_bali_20261114.csv

Folder data/processed:
- openstreetmap_bali_bersih_20261114.csv

Folder docs:
- data_ethics_checklist.csv
- provenance.json

File notebook:
- Untitled7 (1).ipynb

File README:
- README.md


In [ ]:
import shutil
from pathlib import Path

folder_utama = Path("rtm1_2501010235")

# Menyalin notebook dan mengganti namanya
shutil.copy2(
    "Untitled7 (1).ipynb",
    folder_utama / "notebooks" / "akuisisi_openstreetmap_bali.ipynb"
)

# Menyalin data mentah
shutil.copy2(
    "data/raw/openstreetmap_bali_20261114.csv",
    folder_utama / "data/raw/"
)

# Menyalin data bersih
shutil.copy2(
    "data/processed/openstreetmap_bali_bersih_20261114.csv",
    folder_utama / "data/processed/"
)

# Menyalin dokumen
shutil.copy2(
    "docs/provenance.json",
    folder_utama / "docs/"
)

shutil.copy2(
    "docs/data_ethics_checklist.csv",
    folder_utama / "docs/"
)

# Menyalin README
shutil.copy2(
    "README.md",
    folder_utama / "README.md"
)

print("Semua file berhasil disalin ke folder utama!")

Semua file berhasil disalin ke folder utama!


In [ ]:
from pathlib import Path

folder = Path("rtm1_2501010235")

for item in sorted(folder.rglob("*")):
    if item.is_dir():
        print("📁", item.relative_to(folder))
    else:
        print("📄", item.relative_to(folder))

📄 README.md
📁 data
📁 data/processed
📄 data/processed/openstreetmap_bali_bersih_20261114.csv
📁 data/raw
📄 data/raw/openstreetmap_bali_20261114.csv
📁 docs
📄 docs/data_ethics_checklist.csv
📄 docs/provenance.json
📁 notebooks
📄 notebooks/akuisisi_openstreetmap_bali.ipynb


In [ ]:
import json
import shutil
from pathlib import Path

# Buka file provenance
p = Path("docs/provenance.json")

with p.open("r", encoding="utf-8") as f:
    prov = json.load(f)

# Tambahkan identitas pengambil data
prov["pengambil_data"] = "Fransiska Cerlis Oda"
prov["nim"] = "2501010235"

# Simpan perubahan
with p.open("w", encoding="utf-8") as f:
    json.dump(prov, f, indent=4, ensure_ascii=False)

# Salin versi terbaru ke folder tugas
shutil.copy2(
    p,
    Path("rtm1_2501010235/docs/provenance.json")
)

print("Provenance berhasil dilengkapi!")

Provenance berhasil dilengkapi!


In [ ]:
import shutil

zip_path = shutil.make_archive(
    "rtm1_2501010235",
    "zip",
    root_dir=".",
    base_dir="rtm1_2501010235"
)

print("ZIP berhasil dibuat:", zip_path)

ZIP berhasil dibuat: /content/rtm1_2501010235.zip


In [ ]:
from google.colab import files

files.download("/content/rtm1_2501010235.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import zipfile

zip_path = "/content/rtm1_2501010235.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    for nama_file in zip_ref.namelist():
        print(nama_file)

rtm1_2501010235/
rtm1_2501010235/data/
rtm1_2501010235/docs/
rtm1_2501010235/notebooks/
rtm1_2501010235/README.md
rtm1_2501010235/notebooks/akuisisi_openstreetmap_bali.ipynb
rtm1_2501010235/docs/data_ethics_checklist.csv
rtm1_2501010235/docs/provenance.json
rtm1_2501010235/data/processed/
rtm1_2501010235/data/raw/
rtm1_2501010235/data/processed/openstreetmap_bali_bersih_20261114.csv
rtm1_2501010235/data/raw/openstreetmap_bali_20261114.csv
